**Human Movement Intelligence | Biomechanical Load Estimation | Presented by: Reza Saadatyar (2026)**

**01 | Original Dataset, Source Integrity and Trial Provenance**

**Objective:** Integrate the provenance questions of original Stages 01 and 02: official Figshare v13 identity, source integrity, exact participant membership, trial-record keys, original activity distribution and FE representation completeness.

**Source:** Hollinger et al. (2026), Scientific Data, DOI 10.1038/s41597-026-07429-3, Figshare 30225739 v13. This is the complete six-activity source, distinct from the BoneNet derivative.

**Historical evidence:** 40 participant MAT files, 1,897 original movement records, 1,729 structured or opaque FE fields, and 168 FE-empty records. The new notebook must recompute these numbers; its execution is not yet verified.

**Boundary:** Source identity is not a verification of physical units, numerical FE model readiness, or unseen-subject generalisation.

**Draft protection:** New outputs are isolated under output/_consolidation_drafts/01_source_and_trial. All original notebooks and 01_*/02_* results remain untouched.


**00. Colab Repository Setup**


In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.split("/")[-1].removesuffix(".git")

%cd /content

if not repo_path.exists():
    !git clone -q {repo_url}


# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Environment**


In [ ]:
# ======================== 01. Imports and Environment ========================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("biomechanical_load_estimation")


# ---------------------------- 2. Standard Library ----------------------------
import json
from pathlib import Path


# ----------------- 3. Scientific Computing and Data Analysis -----------------
import pandas as pd


# ------------------- 4. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure

plot_style(font_family="DejaVu Sans", title_size=12.5, title_weight="normal",
    label_size=11.5, label_weight="normal", tick_size=11, legend_size=10,
    figure_dpi=100, figure_size=(9, 4))


# ------------------ 5. Shared and Study Research Utilities -------------------
from zipfile import ZipFile, ZIP_DEFLATED
from essex_figshare import (
    ARTICLE_ID, ARTICLE_VERSION, fetch_version_manifest,
    find_published_file, ensure_source_file, inspect_matlab_archive,
    extract_matlab_files, inspect_subject_mat,
)
from essex_record_audit import (
    audit_workbook_ids, audit_mat_records, summarise_trial_evidence,
)


# ------------------- 6. Data and Research Output Locations -------------------
raw_dir = root / "data" / "raw" / "essex_femoral_neck_loading"
mat_dir = raw_dir / "matlab_records"
historical_output_dir = root / "biomechanical_load_estimation" / "output"
output_dir = historical_output_dir / "_consolidation_drafts" / "01_source_and_trial"
raw_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

print("Figshare article:", ARTICLE_ID, "| version:", ARTICLE_VERSION)
print("Research output directory:", output_dir)


**02. Pinned Source Identity and Published File Manifest**

Retrieve file identifiers, published sizes and checksums from Figshare version 13. These metadata establish which official source files belong to the analysis; the downloaded archive is checked separately.


In [ ]:
# =============== 02. Pinned Figshare Article and File Manifest ===============

# ------------------- 1. Retrieve Official Version Metadata -------------------
article, source_files = fetch_version_manifest()
archive_file = find_published_file(source_files, "MATLAB.zip")
source_table = pd.DataFrame([
    {
        "file_id": item.get("id"),
        "filename": item.get("name"),
        "size_bytes": item.get("size"),
        "download_url": item.get("download_url"),
        "published_md5": item.get("computed_md5") or item.get("supplied_md5"),
    }
    for item in source_files
])
display(source_table)


# -------------------- 2. Record Original Dataset Identity --------------------
metadata = {
    "article_id": ARTICLE_ID,
    "version": ARTICLE_VERSION,
    "title": article.get("title"),
    "doi": article.get("doi"),
    "published_date": article.get("published_date"),
    "license": article.get("license"),
    "archive_file_id": archive_file["id"],
    "archive_filename": archive_file["name"],
    "archive_size_bytes": archive_file["size"],
    "archive_published_md5": archive_file.get("computed_md5") or archive_file.get("supplied_md5"),
}
source_table.to_csv(output_dir / "01_figshare_file_manifest.csv", index=False)
print("Article:", metadata["title"])
print("Pinned archive:", metadata["archive_filename"])
print("Archive size (MiB):", round(metadata["archive_size_bytes"] / 1024**2, 2))


**03. Official Archive Acquisition and ZIP Integrity**

Download `MATLAB.zip`, verify its published size/checksum and local SHA256, and check ZIP member paths and CRC values. Source files are cached under `data/raw/` for reproducible analysis.


In [ ]:
# ============ 03. Controlled Source Download and ZIP Verification ============

# --------------- 1. Download and Verify Official Source Bytes ----------------
archive_path, archive_hashes = ensure_source_file(archive_file, raw_dir)
print("Archive:", archive_path)
print("Verified local SHA256:", archive_hashes["sha256"])


# --------------------- 2. Validate the Complete Archive ----------------------
zip_records = inspect_matlab_archive(archive_path, expected_mat_files=40)
archive_table = pd.DataFrame(zip_records)
display(archive_table.head(10))
archive_table.to_csv(output_dir / "01_matlab_archive_inventory.csv", index=False)
print("Verified MATLAB files:", int(archive_table["is_matlab_file"].sum()))
print("Total uncompressed MiB:", round(archive_table["size_bytes"].sum() / 1024**2, 2))


**04. Participant File Extraction and Source Inventory**

Extract the 40 participant MAT files and identify participants by their original filenames. Compare file-level coverage without assuming sequential IDs, demographic characteristics or trial labels.


In [ ]:
# =========== 04. MATLAB File Inventory and Participant Identifiers ===========

# -------------------- 1. Extract Validated MATLAB Members --------------------
mat_paths = extract_matlab_files(archive_path, mat_dir, zip_records)
participant_ids = [path.stem.lower() for path in mat_paths]
assert len(mat_paths) == len(set(participant_ids)) == 40
assert all(path.stat().st_size > 0 for path in mat_paths)


# ------------------- 2. Inspect Explicit Source Filenames --------------------
participant_table = pd.DataFrame({
    "source_file": [path.name for path in mat_paths],
    "subject_id": participant_ids,
    "file_size_bytes": [path.stat().st_size for path in mat_paths],
})
display(participant_table)
participant_table.to_csv(output_dir / "01_matlab_participant_inventory.csv", index=False)
print("Independent source files:", len(participant_table))
print("IDs are observed from filenames, not generated by range.")


**05. Original MATLAB Record Schema and Field Coverage**

Inspect each original `Data` array and record the available fields and object representations. An object stored as an opaque MATLAB structure is counted as such, not interpreted as a recovered numeric signal.


In [ ]:
# ============== 05. Participant-Level MATLAB Data Schema Audit ===============

# --------------------- 1. Inspect Every Source MAT File ----------------------
audit_rows = []
for source_path in mat_paths:
    audit_rows.append(inspect_subject_mat(source_path))

audit_table = pd.DataFrame(audit_rows).sort_values("subject_id").reset_index(drop=True)
display(audit_table[[
    "subject_id", "record_count", "source_data_shape",
    "first_record_type", "field_names_first_record"
]].head(12))


# ---------------- 2. Save Source Representation and Coverage -----------------
audit_table.to_csv(output_dir / "01_matlab_record_schema_audit.csv", index=False)
print("Participants audited:", len(audit_table))
print("Total MATLAB Data records:", int(audit_table["record_count"].sum()))
print("Record counts by participant:")
display(audit_table[["subject_id", "record_count"]])


**06. FE representation and participant workbook reconciliation**

Distinguish source-empty fields from structured/opaque MATLAB objects; compare all workbook participant IDs with original MAT filenames. Do not treat source summary rows as participants.


In [ ]:
# =========== 06. FE Target Representation and Participant Metadata ===========

# ------------------ 1. Summarise FE Target Representations -------------------
status_cols = [
    "numeric_candidate", "empty", "missing_field",
    "nonfinite_numeric", "matlab_object_unresolved",
    "non_numeric_unresolved",
]
target_table = audit_table[["subject_id", "record_count", *status_cols]].copy()
target_table.to_csv(output_dir / "01_femoral_strain_representation.csv", index=False)
display(target_table)
print("FE representation totals:")
display(target_table[status_cols].sum().rename("records").to_frame())


In [ ]:
workbook_item = find_published_file(source_files, "SubCharacteristics.xlsx")
workbook_path, workbook_hashes = ensure_source_file(workbook_item, raw_dir)
# ============= 03. Source Workbook and Participant Identity Gate =============

# ----------------- 1. Parse Original Participant Identifiers -----------------
mat_ids = sorted(participant_ids)
workbook_audit, verified_subjects = audit_workbook_ids(
    workbook_path, mat_ids
)
display(pd.DataFrame([{
    "recognised_rows": workbook_audit["recognised_participant_rows"],
    "unique_ids": workbook_audit["unique_participant_ids"],
    "duplicate_ids": workbook_audit["duplicate_ids"],
    "source_id_match": workbook_audit["id_alignment_passed"],
}]))
print("Missing workbook IDs:", workbook_audit["mat_ids_not_in_workbook"])
print("Unexpected workbook IDs:", workbook_audit["workbook_ids_not_in_mat"])


# ---------------- 2. Record and Require Valid Source Identity ----------------
verified_subjects.to_csv(
    output_dir / "02_workbook_subject_ids.csv", index=False
)
if not workbook_audit["id_alignment_passed"]:
    raise ValueError(
        "Published workbook and MAT participants disagree. "
        "Do not continue to model construction before resolving provenance."
    )
print("Participant identity gate: PASSED")


**07. Original trial identity and IMU/FE field registry**

Audit every original Data record, verify UpdatedTrialID and subject pairing, preserve distinct file-local keys and classify the true source FE representations. This is not a numeric regression dataset.


In [ ]:
# ============== 04. Original Record and Trial Provenance Audit ===============

# --------------------- 1. Inspect All Source MAT Records ---------------------
records = audit_mat_records(mat_paths)
trial_summary = summarise_trial_evidence(records)
display(records[[
    "subject_id", "local_record_index", "task", "subtask",
    "updated_trial_id", "imu_representation", "fe_representation"
]].head(18))

print("Audited source records:", trial_summary["source_records"])
print("Participants:", trial_summary["source_participants"])
print("Readable UpdatedTrialID fields:", trial_summary["readable_updated_trial_ids"])


# --------------------- 2. Save Original Record Registry ----------------------
records.to_csv(
    output_dir / "02_original_record_registry.csv", index=False
)
print("Duplicate participant/trial IDs:", trial_summary["duplicate_subject_trial_ids"])
print("No inferred source-trial identifiers were created.")


In [ ]:
# =============== 05. IMU and FE Representation Quality Profile ===============

# -------------- 1. Group Source Representations by Participant ---------------
coverage = (
    records.groupby(
        ["subject_id", "imu_representation", "fe_representation"]
    ).size().rename("records").reset_index()
)
coverage.to_csv(
    output_dir / "02_modality_representation.csv", index=False
)
display(coverage.head(18))


# ------------------- 2. Report Source Encoding Categories --------------------
encoding = pd.DataFrame(
    [{"modality": "IMU", "representation": k, "records": int(v)}
     for k, v in trial_summary["imu_encoding_counts"].items()]
    + [{"modality": "FE", "representation": k, "records": int(v)}
       for k, v in trial_summary["fe_encoding_counts"].items()]
)
display(encoding)
encoding.to_csv(
    output_dir / "02_modality_encoding_summary.csv", index=False
)
print("FE waveform units and temporal grid: NOT VERIFIED")


**08. Original activity, participant and FE coverage**

Combine unique descriptive panels rather than duplicating the two older plots. Previous figures and output files remain preserved.


In [ ]:
activity_counts = records.groupby("task").size().sort_values(ascending=False)
activity_counts.to_csv(output_dir / "01_activity_record_counts.csv", header=["records"])
subject_counts = records.groupby("subject_id").size().sort_index()
fe_counts = records["fe_representation"].value_counts()
fig, ax = plt.subplots(3, 1, figsize=(11, 11), constrained_layout=True)
ax[0].bar(subject_counts.index, subject_counts.values)
ax[0].set_title("Source records per participant")
ax[0].tick_params(axis="x", labelrotation=90)
ax[1].bar(activity_counts.index, activity_counts.values)
ax[1].set_title("Source activity distribution")
ax[1].tick_params(axis="x", labelrotation=20)
ax[2].bar(fe_counts.index, fe_counts.values)
ax[2].set_title("FE source representation, not measured strain")
ax[2].tick_params(axis="x", labelrotation=15)
save_figure(fig, output_dir / "01_combined_source_coverage.png")
plt.show()


**09. Integrated provenance and scientific Quality Gate**

Reconcile the newly computed subject/trial registry with the immutable historical Stage 01/02 JSONs and their SHA256 archive fingerprints. A pass establishes source identity only, not numerical strain model readiness.


In [ ]:
assert int(target_table["empty"].sum()) == 168
assert int(audit_table["record_count"].sum()) == 1897
assert len(records) == 1897 and records["local_record_key"].is_unique
assert records["task"].nunique() == 6
prior_01 = json.loads((historical_output_dir / "01_source_provenance_and_quality_gate.json").read_text())
prior_02 = json.loads((historical_output_dir / "02_record_provenance_and_quality_gate.json").read_text())
assert prior_01["archive_sha256"] == archive_hashes["sha256"] == prior_02["archive_sha256"]
assert prior_01["matlab_data_elements"] == prior_02["source_record_summary"]["source_records"] == len(records)
# ============= 07. Scientific Quality Gate and Provenance Record =============

# ----------- 1. Validate Source Participant and Record Accounting ------------
assert ARTICLE_ID == 30225739 and ARTICLE_VERSION == 13
assert workbook_audit["id_alignment_passed"]
assert trial_summary["source_participants"] == 40
assert trial_summary["source_records"] == 1897
assert records["local_record_key"].is_unique
assert set(records["subject_id"]) == set(mat_ids)
assert int(sum(trial_summary["fe_encoding_counts"].values())) == len(records)
assert int(sum(trial_summary["imu_encoding_counts"].values())) == len(records)


# ---------------- 2. Save Accurate Modelling-Readiness Flags -----------------
fe_numeric_count = trial_summary["fe_encoding_counts"].get(
    "numeric_candidate", 0
)
provenance = {
    "source_article_id": ARTICLE_ID,
    "version": ARTICLE_VERSION,
    "archive_sha256": archive_hashes["sha256"],
    "participant_workbook_sha256": workbook_hashes["sha256"],
    "participant_workbook_audit": workbook_audit,
    "source_record_summary": trial_summary,
    "source_registry_gate": "PASSED",
    "consolidation_draft": True,
    "numeric_fe_candidates": int(fe_numeric_count),
    "verified_fe_target_order": False,
    "verified_fe_physical_units": False,
    "verified_imu_to_fe_temporal_alignment": False,
    "verified_complete_trial_mapping": False,
    "model_ready": False,
    "model_training_performed": False,
}
(output_dir / "01_consolidated_source_and_trial_quality_gate.json").write_text(
    json.dumps(provenance, indent=2) + "\n", encoding="utf-8"
)
print("STAGE 02 | SOURCE REGISTRY QUALITY GATE: PASSED")
print("MODEL-READY REGRESSION DATASET: NOT ESTABLISHED")
print("Model training and test evaluation performed: NO")


**10. Draft output validation and evidence packaging**

Only this draft's isolated output folder is packaged. The historical 01 and 02 folders, prior notebooks and their saved scientific results are not changed. Do not claim an executed consolidation until Colab execution and verification.


In [ ]:
expected = ["01_figshare_file_manifest.csv","01_matlab_archive_inventory.csv","01_matlab_participant_inventory.csv","01_matlab_record_schema_audit.csv","01_femoral_strain_representation.csv","02_workbook_subject_ids.csv","02_original_record_registry.csv","02_modality_representation.csv","02_modality_encoding_summary.csv","01_activity_record_counts.csv","01_combined_source_coverage.png","01_consolidated_source_and_trial_quality_gate.json"]
missing = [name for name in expected if not (output_dir / name).is_file()]
if missing: raise FileNotFoundError("Incomplete consolidated evidence: "+", ".join(missing))
archive = Path("/content") / "BoneNet_Consolidated_01_Draft_Outputs.zip"
with ZipFile(archive, "w", ZIP_DEFLATED) as z:
    for name in expected: z.write(output_dir / name, arcname=name)
print("Consolidated evidence files:", len(expected), "| ZIP:", archive)
print("GitHub commit/push by notebook: NO")


**11. Scientific interpretation and independent next step**

The consolidated first study stage establishes source and trial lineage, participant identity and original FE representation. It does not verify FE physical units, temporal model-ready pairing, independence of candidate validation folds or any predictive accuracy. Historical evidence remains preserved. The next distinct topic is MATLAB object decoding and numerical recovery, combining previous Stages 03–05.
